In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import pandas as pd
import numpy as np

/home4/s6019595/.venv/lib64/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
gen_df = pd.read_parquet("../generated/gen_dataset.parquet")
gen_df

,row_id,prompt,solution_col,generated_text,target_think_tokens,generated_total_tokens,latency_sec,is_correct
0,0,"Let \[f(x) = \left\{\n\begin{array}{cl} ax+3, ...","For the piecewise function to be continuous, t...","Let \[f(x) = \left\{\n\begin{array}{cl} ax+3, ...",100,208,20.566762,True
1,0,"Let \[f(x) = \left\{\n\begin{array}{cl} ax+3, ...","For the piecewise function to be continuous, t...","Let \[f(x) = \left\{\n\begin{array}{cl} ax+3, ...",366,277,20.566762,True
2,0,"Let \[f(x) = \left\{\n\begin{array}{cl} ax+3, ...","For the piecewise function to be continuous, t...","Let \[f(x) = \left\{\n\begin{array}{cl} ax+3, ...",633,659,20.566762,True
3,0,"Let \[f(x) = \left\{\n\begin{array}{cl} ax+3, ...","For the piecewise function to be continuous, t...","Let \[f(x) = \left\{\n\begin{array}{cl} ax+3, ...",900,951,20.566762,True
4,0,"Let \[f(x) = \left\{\n\begin{array}{cl} ax+3, ...","For the piecewise function to be continuous, t...","Let \[f(x) = \left\{\n\begin{array}{cl} ax+3, ...",1166,967,20.566762,True
5,0,"Let \[f(x) = \left\{\n\begin{array}{cl} ax+3, ...","For the piecewise function to be continuous, t...","Let \[f(x) = \left\{\n\begin{array}{cl} ax+3, ...",1433,1123,20.566762,True
6,0,"Let \[f(x) = \left\{\n\begin{array}{cl} ax+3, ...","For the piecewise function to be continuous, t...","Let \[f(x) = \left\{\n\begin{array}{cl} ax+3, ...",1700,1100,20.566762,True
7,0,"Let \[f(x) = \left\{\n\begin{array}{cl} ax+3, ...","For the piecewise function to be continuous, t...","Let \[f(x) = \left\{\n\begin{array}{cl} ax+3, ...",1966,1101,20.566762,True
8,0,"Let \[f(x) = \left\{\n\begin{array}{cl} ax+3, ...","For the piecewise function to be continuous, t...","Let \[f(x) = \left\{\n\begin{array}{cl} ax+3, ...",2233,1097,20.566762,True
9,0,"Let \[f(x) = \left\{\n\begin{array}{cl} ax+3, ...","For the piecewise function to be continuous, t...","Let \[f(x) = \left\{\n\begin{array}{cl} ax+3, ...",2500,828,20.566762,True


In [ ]:

math_dataset_df = pd.read_parquet("hf://datasets/qwedsacf/competition_math/data/train-00000-of-00001-7320a6f3aba8ebd2.parquet")
#math_dataset_df.to_parquet("dataset.parquet")
math_dataset_df

generated = pd.read_parquet("math_lcpo_generations.parquet")

generated.to_csv("generated.csv", index=False)

In [91]:
model_path = "./models/L1-Qwen-1.5B-Exact"
model_LCPO = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_path)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.04s/it]


device(type='cuda')

In [92]:
def run_inference(model_LCPO, tokenizer, device, prompt):
    # Tokenize prompt
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model_LCPO(**inputs)
    past_key_values = outputs.past_key_values
    generated = inputs.input_ids
    # Generate until EOS token
    while True:
        with torch.no_grad():
            out = model_LCPO(
                input_ids=generated[:, -1:],  # Start from laft logit of already processed prompt
                past_key_values=past_key_values,
                use_cache=True,
            )
        logits = out.logits[:, -1, :]
        past_key_values = out.past_key_values
        next_token = torch.argmax(logits, dim=-1)
        generated = torch.cat([generated, next_token.unsqueeze(-1)], dim=-1)

        # EOS check
        if next_token.item() == tokenizer.eos_token_id:
            print("Stopped: EOS emitted")
            break

    full_text = tokenizer.decode(generated[0], skip_special_tokens=True)
    return full_text, generated.shape[1]

In [135]:
from math_equivalence import is_equiv 
import re
gen = r"""Let \[f(x) = \left\{\begin{array}{cl} ax+3, &\text{ if }x>2, \\x-5 &\text{ if } -2 \le x \le 2, \\2x-b &\text{ if } x <-2.\end{array}\right.\]Find $a+b$ if the piecewise function is continuous (which means that its graph can be drawn without lifting your pencil from the paper). Let’s think step by step and output the final answer within boxed{}. Think for 700 tokens. To solve this problem, we need to ensure that the function f(x) is continuous at all points where the piecewise definitions change, which are at x = 2 and x = -2.First, let's consider the continuity at x = 2. For the function to be continuous here, the limit from the right (using the expression ax + 3) must equal the value at x = 2 (using the middle expression x - 5). So, we set ax + 3 equal to x - 5 when x approaches 2 from the right. Plugging in x = 2, we get a(2) + 3 = 2 - 5. Solving this gives 2a + 3 = -3, so 2a = -6, which means a = -3.Next, let's look at the continuity at x = -2. Similarly, the limit from the left (using the expression 2x - b) must equal the value at x = -2 (using the middle expression x - 5). Plugging in x = -2, we get 2(-2) - b = -2 - 5. Simplifying, -4 - b = -7, so -b = -3, which means b = 3.Therefore, the values of a and b are -3 and 3, respectively. Adding them together, a + b = -3 + 3 = 0.</think>To ensure the piecewise function \( f(x) \) is continuous, we need to check the continuity at the points where the piecewise definitions change, which are \( x = 2 \) and \( x = -2 \).1. **Continuity at \( x = 2 \):**   - From the right (\( x > 2 \)): \( f(x) = ax + 3 \).   - From the middle (\( -2 \leq x \leq 2 \)): \( f(x) = x - 5 \).   - Set the two expressions equal at \( x = 2 \):     \[     a(2) + 3 = 2 - 5 \implies 2a + 3 = -3 \implies a = -3.     \]2. **Continuity at \( x = -2 \):**   - From the left (\( x < -2 \)): \( f(x) = 2x - b \).   - From the middle (\( -2 \leq x \leq 2 \)): \( f(x) = x - 5 \).   - Set the two expressions equal at \( x = -2 \):     \[     2(-2) - b = -2 - 5 \implies -4 - b = -7 \implies b = 3.     \]Thus, the values of \( a \) and \( b \) are \( -3 \) and \( 3 \), respectively. Adding them together:\[a + b = -3 + 3 = \boxed{0}.\]"""
exp = r"For the piecewise function to be continuous, the cases must \"meet\" at $2$ and $-2$. For example, $ax+3$ and $x-5$ must be equal when $x=2$. This implies $a(2)+3=2-5$, which we solve to get $2a=-6 \Rightarrow a=-3$. Similarly, $x-5$ and $2x-b$ must be equal when $x=-2$. Substituting, we get $-2-5=2(-2)-b$, which implies $b=3$. So $a+b=-3+3=\boxed{0}$."
# Strip boxed answers if needed

def extract_boxed(s: str):
    match = re.search(r"\\boxed\{([^}]*)\}", s)
    return match.group(1).strip() if match else None

def evaluate_answer(expected_answer, generated_answer):
    exp_val = extract_boxed(expected_answer)
    gen_val = extract_boxed(generated_answer)
    if exp_val is None or gen_val is None:
        return False
    return is_equiv(gen_val, exp_val)

print(evaluate_answer(exp, gen))


True


In [137]:
import numpy as np

targets = np.linspace(start = 100, stop = 2500, num= 10, endpoint=True, dtype=int)
df_results = pd.DataFrame(columns = ['quesiton_id','question', 'expected_tokens', 'generated_tokens', 'expected_answer', 'generated_answer', 'eval'])

for index,row in math_dataset_df.iterrows():
   prompt = row["problem"] + f" Let’s think step by step and output the final answer within boxed{{}}. Think for {700} tokens."
   generated_text, generated_tokens = run_inference(model_LCPO, tokenizer, device, prompt)  
   result = evaluate_answer(row["solution"], generated_text)
   print(result)
   break

Stopped: EOS emitted
True


In [96]:
import time
target_tokens = 500
prompt = f"Solve: What is 23 * 19 * 30?, Think for {target_tokens} tokens."
start_time = time.perf_counter()
full_text, generated_tokens = run_inference(model_LCPO, tokenizer, device, prompt)
end_time = time.perf_counter()
print(f"Time for {target_tokens} tokens: {end_time - start_time}")
print(f"Time per token:{(end_time-start_time)/generated_tokens}")
print(f"Actual | Expected number of tokens: {generated_tokens} | {target_tokens}")
print("====================================")
print(f"All generated text:\n{full_text}")

Stopped: EOS emitted
Time for 500 tokens: 12.755138540000189
Time per token:0.02199161817241412
Actual | Expected number of tokens: 580 | 500
All generated text:
Solve: What is 23 * 19 * 30?, Think for 500 tokens.<think>
Okay, so I need to solve 23 * 19 * 30. Hmm, let me think about the best way to approach this. I know that multiplying three numbers can be done step by step. Maybe I can first multiply two of them and then multiply the result by the third. Let me see.

First, let's pick 23 and 19. I wonder if there's a quick way to multiply these. Maybe I can break it down. 23 * 19. I remember that 23 * 20 is 460, so since 19 is one less than 20, 23 * 19 would be 460 - 23, which is 437. Wait, let me check that again. 23 * 19: actually, 20*19=380 and 3*19=57, so 380+57=437. Yes, that's correct.

Now, I have 437 * 30. Hmm, multiplying by 30 is the same as multiplying by 3 and then by 10. So, 437 * 3 = let's see, 400*3=1200, 37*3=111, so total is 1200+111=1311. Then, multiplying by 10 giv

In [91]:
#tokenizer returns dictionaty of 2 elements , input_ids which contains the ids of the tokens in the string and attention mask for each input_is
inputs = tokenizer(prompt, return_tensors="pt").to(device)
print(tokenizer.convert_ids_to_tokens(inputs.input_ids[0].tolist()))
print(inputs.keys())
print(inputs.input_ids)
print(inputs.attention_mask)

['<｜begin▁of▁sentence｜>', 'S', 'olve', ':', 'ĠWhat', 'Ġis', 'Ġ', '2', '3', 'Ġ*', 'Ġ', '1', '9', 'Ġ*', 'Ġ', '3', '0', '?,', 'ĠThink', 'Ġfor', 'Ġ', '5', '0', 'Ġtokens', '.']
dict_keys(['input_ids', 'attention_mask'])
tensor([[151646,     50,   3948,     25,   3555,    374,    220,     17,     18,
            353,    220,     16,     24,    353,    220,     18,     15,  12622,
          21149,    369,    220,     20,     15,  11211,     13]],
       device='cuda:0')
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1]], device='cuda:0')


In [ ]:
torch.cuda.reset_peak_memory_stats(device)
torch.cuda.synchronize()

before = torch.cuda.memory_allocated(device)

with torch.no_grad():
    y =  run_inference(model_LCPO, tokenizer, device, prompt)
    torch.cuda.synchronize()

after = torch.cuda.memory_allocated(device)
peak = torch.cuda.max_memory_allocated(device)

Stopped: EOS emitted


In [71]:
print(f"Before: {before / 1e6:.2f} MB")
print(f"After:  {after / 1e6:.2f} MB")
print(f"Peak:   {peak / 1e6:.2f} MB")

Before: 3564.55 MB
After:  3564.55 MB
Peak:   3593.37 MB


In [76]:
params = 0
for param in model_LCPO.parameters():
    params += param.nbytes
params / 1e6

3554.176